# Eigenmode simulation of coupled qubits

## Rendering the design

## Two qubit example with resonators and a coupler


In [ ]:
%load_ext autoreload
%autoreload 2

import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

import mini_studies as ms
import optimization_targets as ot

import parameter_targets as pt
import plot_settings as ps

from qdesignoptimizer.design_analysis import DesignAnalysis, DesignAnalysisState
from qdesignoptimizer.utils.utils import get_save_path
from qdesignoptimizer.utils.names_parameters import param, param_nonlin

from qdesignoptimizer.utils.utils import close_ansys

close_ansys()

design, gui = create_chip_base(n.CHIP_NAME, d.chip_type, open_gui=True)
d.render_qiskit_metal_design(design, gui)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Running the study in partitions 


In [ ]:
# Iteratively optimize the two qubit chip with the following divisions:

RES_QB_CPLR = "res_qb_cplr"
QB_CPLR_QB = "qb_cplr_qb"
CPLR_QB_RES = "cplr_qb_res"

nbr_iterations = 15
group_passes = 10
delta_f = 0.001

counter = 0
optimization_results = []
minimization_results = []

PLOT_SETTINGS = ps.PLOT_SETTINGS_TWO_QB
RENDER_QISKIT_METAL = lambda design: d.render_qiskit_metal_design(design, gui)
design_analysis_state = DesignAnalysisState(
    design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS
)

for i in range(nbr_iterations):
    for section in [RES_QB_CPLR, CPLR_QB_RES, QB_CPLR_QB]:

        MINI_STUDY = ms.get_mini_study_2qb_resonator_coupler_partitioned(section)

        if section == RES_QB_CPLR:
            groups = [n.NBR_1]
            exclude_param_from_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_1),
            ]
        elif section == CPLR_QB_RES:
            groups = [n.NBR_2]
            exclude_param_from_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_2),
            ]
        elif section == QB_CPLR_QB:
            groups = [n.NBR_1, n.NBR_2]
            exclude_param_from_update = [
                param(n.QUBIT_1, n.FREQ),
                param_nonlin(n.QUBIT_1, n.QUBIT_1),
                param(n.QUBIT_2, n.FREQ),
                param_nonlin(n.QUBIT_2, n.QUBIT_2),
            ]
        design_analysis_state.exclude_param_from_update = exclude_param_from_update
        opt_targets = ot.get_opt_targets_2qubits_resonator_coupler(
            groups=groups,
            opt_target_qubit_freq=section in [RES_QB_CPLR, CPLR_QB_RES],
            opt_target_qubit_anharm=section in [RES_QB_CPLR, CPLR_QB_RES],
            opt_target_resonator_freq=section in [RES_QB_CPLR, CPLR_QB_RES],
            opt_target_resonator_qubit_chi=section in [RES_QB_CPLR, CPLR_QB_RES],
            opt_target_coupler_freq=section == QB_CPLR_QB,
            opt_target_coupler_anharm=section == QB_CPLR_QB,
            opt_target_coupler_qubit_chi=section == QB_CPLR_QB,
        )

        design_analysis = DesignAnalysis(
            design_analysis_state,
            mini_study=MINI_STUDY,
            opt_targets=opt_targets,
            save_path=get_save_path("out/", n.CHIP_NAME),
            update_design_variables=False,
            plot_settings=PLOT_SETTINGS,
        )
        # Continue from optimization and minimization results from previous partitions
        design_analysis.optimization_results = optimization_results
        design_analysis.minimization_results = minimization_results

        design_analysis.update_nbr_passes(group_passes)
        design_analysis.update_delta_f(delta_f)
        design_analysis.optimize_target({}, {})
        design_analysis.screenshot(gui=gui, run=counter)

        # Store optimization and minimization results for the next iteration
        optimization_results = design_analysis.optimization_results
        minimization_results = design_analysis.minimization_results

        counter += 1

## Results

### Eigenmodes

In [ ]:
design_analysis.get_eigenmode_results()

### Cross Kerr

In [ ]:
design_analysis.get_cross_kerr_matrix(iteration=-1)

## Update parameters

In [ ]:
design_analysis.overwrite_parameters()

## Close